# NYSSIS ↔ StudentID Matching — Reference Notebook

**Domain:** SCSD Education Analytics

The roster preprocessing step that feeds the Charter Audit Engine: fill missing
NYSSIS IDs and resolve Student IDs by matching on First + Last + DOB + Grade.
Documents both the original Excel array formulas and their SQL equivalent.

## A. Excel — fill missing NYSSIS ID (Gen Roster, array formula)
Leaves existing NYSSIS IDs untouched; for blanks, matches First+Last+DOB+Grade
against `Export (28)`. Enter with **Ctrl+Shift+Enter**.

In [ ]:
excel_fill_nyssis = r"""
=IF(A2<>"", A2,
  IFERROR(
    INDEX('Export (28)'!$A:$A,
      MATCH(1,
        ('Export (28)'!$C:$C=D2) *
        ('Export (28)'!$D:$D=C2) *
        (TEXT('Export (28)'!$E:$E,"MM/DD/YYYY")=TEXT(F2,"MM/DD/YYYY")) *
        ('Export (28)'!$F:$F=E2),
        0)
    ),
  "")
)
"""
print(excel_fill_nyssis)

## B. Excel — resolve Student ID by NYSSIS with name/DOB fallback
Runs the original VLOOKUP when NYSSIS exists; falls back to name+DOB+grade match;
returns `"Not Found"` otherwise. Enter with **Ctrl+Shift+Enter**.

In [ ]:
excel_resolve_studentid = r"""
=IF(A2<>"",
  VLOOKUP(A2,'Export (28)'!$A:$B,2,0),
  IFERROR(
    INDEX('Export (28)'!$B:$B,
      MATCH(1,
        ('Export (28)'!$C:$C=D2)*
        ('Export (28)'!$D:$D=C2)*
        (TEXT('Export (28)'!$E:$E,"MM/DD/YYYY")=TEXT(F2,"MM/DD/YYYY"))*
        ('Export (28)'!$F:$F=E2),
        0)
    ),
  "Not Found")
)
"""
print(excel_resolve_studentid)

## C. SQL equivalent — composite-key match
Same logic as the Excel fallback, expressed as a LEFT JOIN on the composite key
(first + last + DOB + grade) to recover NYSSIS / Student ID where the direct key
is missing.

In [ ]:
%%sql
SELECT
    g.*,
    COALESCE(g.nyssis_id, e.nyssis_id)         AS nyssis_id_filled,
    COALESCE(g.student_id, e.student_id, 'Not Found') AS student_id_resolved
FROM gen_roster g
LEFT JOIN export_28 e
       ON  UPPER(TRIM(e.first_name)) = UPPER(TRIM(g.first_name))
       AND UPPER(TRIM(e.last_name))  = UPPER(TRIM(g.last_name))
       AND CAST(e.dob AS DATE)       = CAST(g.dob AS DATE)
       AND e.grade                   = g.grade;